In [ ]:
# Colab: 여러 CSV(예: 9개)에서 '한국 거래일(2018-01-02 ~ 2019-02-28)'만 남기기 + ZIP 다운로드
!pip -q install exchange_calendars

import io, os, zipfile, shutil
import pandas as pd
from google.colab import files
import exchange_calendars as ecals

# -----------------------------
# 1) 기간 설정
# -----------------------------
START = "2018-01-02"
END   = "2019-02-28"

# -----------------------------
# 2) CSV 여러 개 업로드
# -----------------------------
uploaded = files.upload()  # 9개 모두 선택해서 업로드

# -----------------------------
# 3) 유연한 CSV 읽기 & 날짜 파싱
# -----------------------------
def read_csv_guess(content: bytes) -> pd.DataFrame:
    for enc in ("cp949", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=enc)
        except Exception:
            pass
    # 마지막 시도: 기본 옵션
    return pd.read_csv(io.BytesIO(content))

def parse_kor_date(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        for fmt in ("%Y.%m.%d", "%Y-%m-%d", "%Y/%m/%d"):
            dt = dt.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return dt

# -----------------------------
# 4) KRX 거래일 캘린더
# -----------------------------
xkrx = ecals.get_calendar("XKRX")
sessions = xkrx.sessions_in_range(START, END)           # 실제 거래일(주말·공휴일·임시휴장 제외)
allowed_dates = set(pd.DatetimeIndex(sessions).date)    # date 객체 집합

print(f"기간 내 거래일 수: {len(sessions)} (예상: 283)")

# -----------------------------
# 5) 파일별 필터링 처리
# -----------------------------
os.makedirs("filtered_outputs", exist_ok=True)
summary_rows = []

for fname, content in uploaded.items():
    try:
        df = read_csv_guess(content)
    except Exception as e:
        print(f"[실패] {fname}: CSV 읽기 실패 -> {e}")
        continue

    if "날짜" not in df.columns:
        print(f"[건너뜀] {fname}: '날짜' 컬럼이 없습니다.")
        continue

    dt = parse_kor_date(df["날짜"])
    if dt.isna().all():
        print(f"[건너뜀] {fname}: '날짜' 파싱 실패(포맷 확인 필요)")
        continue

    df["__date__"] = dt.dt.date
    mask = df["__date__"].isin(allowed_dates)
    kept = df.loc[mask].copy()

    # 정렬(네이버 CSV가 보통 최신→과거 순이라 오름차순으로 정렬)
    kept["날짜_dt"] = parse_kor_date(kept["날짜"])
    kept = kept.sort_values("날짜_dt").drop(columns=["날짜_dt", "__date__"])

    outpath = os.path.join("filtered_outputs", f"filtered_{fname}")
    kept.to_csv(outpath, index=False, encoding="utf-8-sig")

    summary_rows.append({
        "file": fname,
        "original_rows": len(df),
        "kept_rows(trading_days)": len(kept),
        "output_path": outpath
    })
    print(f"[완료] {fname}: 원본 {len(df)}행 -> 거래일 {len(kept)}행")

# 요약 저장
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("filtered_outputs/summary.csv", index=False, encoding="utf-8-sig")
print("\n=== 처리 요약 ===")
print(summary_df)

# -----------------------------
# 6) ZIP으로 묶어서 한 번에 다운로드
# -----------------------------
shutil.make_archive("filtered_outputs", "zip", "filtered_outputs")
from google.colab import files as colab_files
colab_files.download("filtered_outputs.zip")


ERROR: Operation cancelled by user


ModuleNotFoundError: No module named 'exchange_calendars'

In [ ]:
# ======================================
# Excel Merger + Sort by Date (ascending)
# ======================================
from pathlib import Path
import pandas as pd
import re

# ===== 설정 =====
FILE_PATHS = [
    "/content/현대차 2군 1.xlsx",
    "/content/현대차 2군 2.xlsx"
]  # 또는 폴더에서 *.xlsx 읽어와도 됩니다.
DATE_CANDIDATES = ["일자", "날짜", "거래일", "기준일", "date", "Date", "DATE"]
ADD_SOURCE_COLUMN = True
OUTPUT_CSV = "./merged_sorted.csv"
OUTPUT_XLSX = "./merged_sorted.xlsx"

# ===== 유틸 =====
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = []
    for c in df.columns:
        c2 = str(c).replace("\n", " ").strip()
        c2 = re.sub(r"\s+", " ", c2)
        cols.append(c2)
    df.columns = cols
    return df

def find_date_col(df: pd.DataFrame, candidates) -> str | None:
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    # 후보가 없으면 파싱 가능한 열을 휴리스틱으로 탐색
    best, best_nonnull = None, -1
    for c in df.columns:
        try:
            parsed = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
            nonnull = parsed.notna().sum()
            if nonnull > best_nonnull and nonnull >= 3:
                best, best_nonnull = c, nonnull
        except Exception:
            pass
    return best

# ===== 메인 =====
frames = []
for path in map(Path, FILE_PATHS):
    df = pd.read_excel(path)
    df = normalize_columns(df)
    date_col = find_date_col(df, DATE_CANDIDATES)
    if date_col is None:
        raise KeyError(f"'{path.name}'에서 날짜 열을 찾지 못했습니다. 열 이름을 확인해주세요.")
    df["_정렬일자"] = pd.to_datetime(df[date_col], errors="coerce", infer_datetime_format=True)
    if ADD_SOURCE_COLUMN:
        df.insert(0, "source_file", path.name)
    frames.append(df)

merged = pd.concat(frames, ignore_index=True, sort=False)
merged = merged.sort_values("_정렬일자").reset_index(drop=True)

# 저장
merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as w:
    merged.to_excel(w, index=False)

print("완료:", merged.shape, "저장 ->", OUTPUT_CSV, OUTPUT_XLSX)


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/tmp/ipython-input-2272189641.py:53: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["_정렬일자"] = pd.to_datetime(df[date_col], errors="coerce", infer_datetime_format=True)
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/tmp/ipython-input-2272189641.py:53: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is no

완료: (29401, 21) 저장 -> ./merged_sorted.csv ./merged_sorted.xlsx


In [ ]:
import pandas as pd
import os
from google.colab import files

# 1. 엑셀 파일 업로드
uploaded = files.upload()  # 파일 선택창 뜸

# 업로드한 파일 이름 가져오기
excel_file = list(uploaded.keys())[0]

# 2. 출력 폴더 생성
output_dir = "output_csv"
os.makedirs(output_dir, exist_ok=True)

# 3. 모든 시트 → CSV 변환
xls = pd.ExcelFile(excel_file)
for sheet_name in xls.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    csv_filename = f"{sheet_name}.csv"
    output_path = os.path.join(output_dir, csv_filename)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ {csv_filename} 저장 완료!")

print("📂 모든 시트를 CSV로 변환 완료!")

# 4. 변환된 파일들 압축해서 다운로드
import shutil
shutil.make_archive("csv_files", 'zip', output_dir)
files.download("csv_files.zip")


Saving KB금융 1군.xlsx to KB금융 1군.xlsx
Saving KB금융 2군.xlsx to KB금융 2군.xlsx
Saving LG화학 1군.xlsx to LG화학 1군.xlsx
Saving LG화학 2군.xlsx to LG화학 2군.xlsx
Saving POSCO홀딩스 1군.xlsx to POSCO홀딩스 1군.xlsx
Saving POSCO홀딩스 2군.xlsx to POSCO홀딩스 2군.xlsx
Saving SK하이닉스 1군.xlsx to SK하이닉스 1군.xlsx
Saving SK하이닉스 2군.xlsx to SK하이닉스 2군.xlsx
Saving 기아 1군.xlsx to 기아 1군.xlsx
Saving 기아 2군.xlsx to 기아 2군.xlsx
Saving 네이버 1군.xlsx to 네이버 1군.xlsx
Saving 네이버 2군.csv to 네이버 2군.csv
Saving 삼성바이오로직스 1군.xlsx to 삼성바이오로직스 1군.xlsx
Saving 삼성바이오로직스 2군.xlsx to 삼성바이오로직스 2군.xlsx
Saving 삼성전자 1군.xlsx to 삼성전자 1군.xlsx
Saving 삼성전자 2군.csv to 삼성전자 2군.csv
Saving 현대자동차 1군.xlsx to 현대자동차 1군.xlsx
Saving 현대차 2군.csv to 현대차 2군.csv
✅ sheet.csv 저장 완료!
📂 모든 시트를 CSV로 변환 완료!


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

In [ ]:
directory_path = '/content/drive/MyDrive/뉴스기사/'

In [ ]:
import os
import pandas as pd

directory_path = "/content/drive/MyDrive/뉴스기사"  # 👉 여기에 CSV 파일이 있는 폴더 경로 입력

try:
    # 디렉토리 내의 모든 파일 목록 가져오기
    all_files = os.listdir(directory_path)
    # .csv로 끝나는 파일만 필터링
    csv_files = [f for f in all_files if f.endswith('.csv')]

    if not csv_files:
        print(f"❌ 오류: '{directory_path}' 디렉토리에서 CSV 파일을 찾을 수 없습니다.")
    else:
        print(f"총 {len(csv_files)}개의 CSV 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            file_path = os.path.join(directory_path, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            # CSV 파일 읽기 (여러 인코딩 시도)
            df = None
            for enc in ["utf-8", "cp949", "euc-kr"]:
                try:
                    df = pd.read_csv(file_path, encoding=enc)
                    print(f"   ✅ 인코딩 성공: {enc}")
                    break
                except UnicodeDecodeError:
                    continue

            if df is None:
                print(f"❌ 인코딩 불가: {file_name}")
                continue

            # '일자' 열을 기준으로 기사 수 계산
            if "일자" not in df.columns:
                print(f"❌ 오류: '{file_name}' 파일에 '일자' 열이 없습니다.")
                continue

            article_counts = df['일자'].value_counts().reset_index()
            article_counts.columns = ['A', 'B']

            # 날짜(A열) 기준으로 오름차순 정렬
            article_counts = article_counts.sort_values(by='A').reset_index(drop=True)

            # 새 파일명으로 저장
            name, ext = os.path.splitext(file_name)
            new_filename = f"{name}_뉴스기사수{ext}"
            output_path = os.path.join(directory_path, new_filename)


            article_counts.to_csv(output_path, index=False, encoding="utf-8-sig")
            print(f"✅ 완료! -> 저장된 파일: {new_filename}\n")

        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError:
    print(f"❌ 오류: 디렉토리를 찾을 수 없습니다. '{directory_path}' 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


❌ 오류: '/content/drive/MyDrive/뉴스기사' 디렉토리에서 CSV 파일을 찾을 수 없습니다.


위에 코드를 진행하면 '/content/drive/MyDrive/뉴스기사' 에 다 저장되는데 수기로 1군_뉴스기사수,2군_뉴스기사수 폴더 만들어서 이동시킨후 밑에 코드진행

In [ ]:
#=======================================================================
# ✅ 사용자 설정 영역
# 입력 폴더와 출력 폴더 경로를 설정합니다.
# =======================================================================
# 기사 수가 집계된 파일들이 있는 폴더
input_directory = '/content/drive/MyDrive/뉴스기사/1군_뉴스기사수/'

# 3배 처리된 파일을 저장할 폴더
output_directory = '/content/drive/MyDrive/뉴스기사/triple/'
# =======================================================================


# STEP 3: 출력 디렉토리 생성
# 저장할 폴더가 없으면 자동으로 생성합니다.
os.makedirs(output_directory, exist_ok=True)
print(f"결과가 저장될 폴더: {output_directory}")
print("-" * 50)

결과가 저장될 폴더: /content/drive/MyDrive/뉴스기사/triple/
--------------------------------------------------


In [ ]:
# STEP 4: 지정된 디렉토리의 모든 CSV 파일 처리
try:
    # 입력 디렉토리 내의 모든 파일 목록 가져오기
    all_files = os.listdir(input_directory)
    # .csv로 끝나는 파일만 필터링
    csv_files = [f for f in all_files if f.endswith('.csv')]

    if not csv_files:
        print(f"❌ 오류: '{input_directory}' 디렉토리에서 CSV 파일을 찾을 수 없습니다.")
    else:
        print(f"총 {len(csv_files)}개의 CSV 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            # 파일 전체 경로
            file_path = os.path.join(input_directory, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            # CSV 파일 읽기
            df = pd.read_csv(file_path)

            # 'B' 열(기사 수)에 3을 곱하기
            # 'B'열이 있는지 확인하여 오류 방지
            if 'B' in df.columns:
                df['B'] = df['B'] * 3
            else:
                print(f"  - ⚠️ 경고: '{file_name}' 파일에 'B' 열이 없어 건너뜁니다.")
                continue

            # STEP 5: 새 파일명으로 저장
            name, ext = os.path.splitext(file_name)
            new_filename = f"{name}_triple{ext}"
            output_path = os.path.join(output_directory, new_filename)

            # 새 CSV 파일로 저장
            df.to_csv(output_path, index=False)
            print(f"  - ✅ 완료! -> 저장된 파일: {new_filename}\n")

        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError:
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. '{input_directory}' 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


총 9개의 CSV 파일을 처리합니다.
--------------------------------------------------
🔄 처리 중인 파일: SK하이닉스 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: SK하이닉스 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: POSCO홀딩스 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: POSCO홀딩스 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: LG화학 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: LG화학 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: KB금융 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: KB금융 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: 네이버 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: 네이버 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: 기아 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: 기아 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: 삼성바이오로직스 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: 삼성바이오로직스 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: 삼성전자 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: 삼성전자 1군_뉴스기사수_triple.csv

🔄 처리 중인 파일: 현대차 1군_뉴스기사수.csv
  - ✅ 완료! -> 저장된 파일: 현대차 1군_뉴스기사수_triple.csv

---------

SUM 코드

In [ ]:
# =======================================================================
# ✅ 사용자 설정 영역
# 1군, 2군 데이터가 있는 폴더와 결과를 저장할 폴더 경로를 설정합니다.
# =======================================================================
# 1군(가중치 적용) 데이터 폴더
dir1 = '/content/drive/MyDrive/뉴스기사/triple/'
# 2군 데이터 폴더
dir2 = '/content/drive/MyDrive/뉴스기사/2군_뉴스기사수/'
# 결과를 저장할 폴더
output_dir = '/content/drive/MyDrive/뉴스기사/SUM/'
# =======================================================================

# STEP 3: 출력 디렉토리 생성
# 저장할 폴더가 없으면 자동으로 생성합니다.
os.makedirs(output_dir, exist_ok=True)
print(f"결과가 저장될 폴더: {output_dir}")
print("-" * 50)

결과가 저장될 폴더: /content/drive/MyDrive/뉴스기사/SUM/
--------------------------------------------------


In [ ]:
try:
    # 각 폴더에 있는 CSV 파일 목록을 가져옵니다.
    files_in_dir1 = [f for f in os.listdir(dir1) if f.endswith('.csv')]
    files_in_dir2 = [f for f in os.listdir(dir2) if f.endswith('.csv')]

    # 2군 파일들을 회사 이름(키)과 파일 경로(값)로 매핑하는 딕셔너리 생성
    # 예: {'기아': '/path/to/기아 2군_뉴스기사수.csv'}
    dir2_map = {f.split(' ')[0]: os.path.join(dir2, f) for f in files_in_dir2}

    if not files_in_dir1:
        print(f"❌ 오류: '{dir1}' 디렉토리에서 처리할 파일을 찾을 수 없습니다.")
    else:
        # 1군 데이터 폴더의 파일 목록을 기준으로 작업 시작
        for file1_name in files_in_dir1:
            # 회사 이름 추출 (첫 번째 공백 앞부분)
            company_key = file1_name.split(' ')[0]
            print(f"🔄 '{company_key}' 회사 처리 시작...")

            # 2군 맵에서 짝이 되는 파일 찾기
            if company_key in dir2_map:
                file1_path = os.path.join(dir1, file1_name)
                file2_path = dir2_map[company_key] # 맵에서 2군 파일 경로 가져오기

                # 데이터프레임으로 파일 읽기
                df1 = pd.read_csv(file1_path)
                df2 = pd.read_csv(file2_path)

                # 'A'열(날짜)을 기준으로 외부 조인(outer merge) 수행
                merged_df = pd.merge(df1, df2, on='A', how='outer', suffixes=('_1', '_2'))

                # 'B'열(기사 수) 합산. NaN(결측값)은 0으로 처리 후 더하기
                merged_df['B'] = merged_df['B_1'].fillna(0) + merged_df['B_2'].fillna(0)

                # 필요한 열만 선택 (A: 날짜, B: 합산된 기사 수)
                # 날짜 순으로 정렬
                final_df = merged_df[['A', 'B']].sort_values(by='A').reset_index(drop=True)

                # 저장할 파일명 생성 (예: '기아_sum.csv')
                output_filename = f"{company_key}_sum.csv"
                output_path = os.path.join(output_dir, output_filename)

                # CSV 파일로 저장
                final_df.to_csv(output_path, index=False)
                print(f"  - ✅ 완료! -> 저장된 파일: {output_filename}\n")

            else:
                print(f"  - ⚠️ 경고: 짝이 되는 2군 파일을 찾을 수 없어 '{company_key}' 건너뜁니다.\n")

        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError:
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


🔄 'SK하이닉스' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: SK하이닉스_sum.csv

🔄 'POSCO홀딩스' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: POSCO홀딩스_sum.csv

🔄 'LG화학' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: LG화학_sum.csv

🔄 'KB금융' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: KB금융_sum.csv

🔄 '네이버' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: 네이버_sum.csv

🔄 '기아' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: 기아_sum.csv

🔄 '삼성바이오로직스' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: 삼성바이오로직스_sum.csv

🔄 '삼성전자' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: 삼성전자_sum.csv

🔄 '현대차' 회사 처리 시작...
  - ✅ 완료! -> 저장된 파일: 현대차_sum.csv

--------------------------------------------------
🎉 모든 파일 처리가 완료되었습니다!


휴장일 처리코드

In [ ]:
# =======================================================================
# ✅ 사용자 설정 영역
# 입력 폴더와 출력 폴더 경로를 설정합니다.
# =======================================================================
# 합산된 데이터가 있는 폴더
input_directory = '/content/drive/MyDrive/뉴스기사/SUM/'
# 휴장일 정보를 추가하여 저장할 폴더
output_directory = '/content/drive/MyDrive/뉴스기사/holiday/'
# =======================================================================

# STEP 3: 휴장일 정보 설정 (2025년 기준)
# 주말(토, 일)은 코드에서 자동으로 휴장일 처리됩니다.
holiday_set = set([
    # 2025년
    '2018-01-01',  # 신정
'2018-02-15', '2018-02-16',  # 설 연휴
'2018-03-01',  # 삼일절
'2018-05-01',  # 근로자의 날
'2018-05-07',  # 어린이날 대체휴일
'2018-05-22',  # 석가탄신일
'2018-06-06',  # 현충일
'2018-06-13',  # 지방선거일
'2018-08-15',  # 광복절
'2018-09-24', '2018-09-25', '2018-09-26',  # 추석 연휴
'2018-10-03',  # 개천절
'2018-10-09',  # 한글날
'2018-12-25',  # 성탄절
'2018-12-31' # 연말 휴장

])

In [ ]:
os.makedirs(output_directory, exist_ok=True)
print(f"결과가 저장될 폴더: {output_directory}")
print("-" * 50)

결과가 저장될 폴더: /content/drive/MyDrive/뉴스기사/holiday/
--------------------------------------------------


In [ ]:
try:
    # 입력 디렉토리 내의 모든 CSV 파일 목록 가져오기
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]

    if not csv_files:
        print(f"❌ 오류: '{input_directory}' 디렉토리에서 CSV 파일을 찾을 수 없습니다.")
    else:
        print(f"총 {len(csv_files)}개의 CSV 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            file_path = os.path.join(input_directory, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            # CSV 파일 읽기
            df = pd.read_csv(file_path)

            # 'A'열을 날짜 형식으로 변환 (YYYYMMDD 포맷을 명시적으로 지정)
            df['Date_Parsed'] = pd.to_datetime(df['A'], format='%Y%m%d', errors='coerce')

            # C열: 요일 추가 (영어 이름으로 저장)
            df['C'] = df['Date_Parsed'].dt.day_name()

            # D열: 휴장일 여부 추가 (주말 또는 공휴일이면 1, 아니면 0)
            df['D'] = df['Date_Parsed'].apply(
                lambda x: 1 if pd.isnull(x) or (x.strftime('%Y-%m-%d') in holiday_set or x.weekday() >= 5) else 0
            )

            # 임시로 사용한 날짜 열 제거
            df = df.drop(columns=['Date_Parsed'])

            # 열 순서 정리: A(날짜), B(기사수), C(요일), D(휴장일여부)
            df = df[['A', 'B', 'C', 'D']]

            # 새 파일명 생성 (예: 'KB금융_sum_holiday.csv')
            name, ext = os.path.splitext(file_name)
            new_filename = f"{name}_holiday{ext}"
            output_path = os.path.join(output_directory, new_filename)

            # 새 CSV 파일로 저장
            df.to_csv(output_path, index=False)
            print(f"  - ✅ 완료! -> 저장된 파일: {new_filename}\n")

        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError:
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. '{input_directory}' 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


총 9개의 CSV 파일을 처리합니다.
--------------------------------------------------
🔄 처리 중인 파일: SK하이닉스_sum.csv
  - ✅ 완료! -> 저장된 파일: SK하이닉스_sum_holiday.csv

🔄 처리 중인 파일: POSCO홀딩스_sum.csv
  - ✅ 완료! -> 저장된 파일: POSCO홀딩스_sum_holiday.csv

🔄 처리 중인 파일: LG화학_sum.csv
  - ✅ 완료! -> 저장된 파일: LG화학_sum_holiday.csv

🔄 처리 중인 파일: KB금융_sum.csv
  - ✅ 완료! -> 저장된 파일: KB금융_sum_holiday.csv

🔄 처리 중인 파일: 네이버_sum.csv
  - ✅ 완료! -> 저장된 파일: 네이버_sum_holiday.csv

🔄 처리 중인 파일: 기아_sum.csv
  - ✅ 완료! -> 저장된 파일: 기아_sum_holiday.csv

🔄 처리 중인 파일: 삼성바이오로직스_sum.csv
  - ✅ 완료! -> 저장된 파일: 삼성바이오로직스_sum_holiday.csv

🔄 처리 중인 파일: 삼성전자_sum.csv
  - ✅ 완료! -> 저장된 파일: 삼성전자_sum_holiday.csv

🔄 처리 중인 파일: 현대차_sum.csv
  - ✅ 완료! -> 저장된 파일: 현대차_sum_holiday.csv

--------------------------------------------------
🎉 모든 파일 처리가 완료되었습니다!


요일 합치고 정규화

In [ ]:
# =======================================================================
# ✅ 사용자 설정 영역
# 입력 폴더와 출력 폴더 경로를 설정합니다.
# =======================================================================
# 휴장일 정보가 포함된 파일들이 있는 폴더
input_directory = '/content/drive/MyDrive/뉴스기사/holiday/'
# 정규화된 최종 결과를 저장할 폴더
output_directory = '/content/drive/MyDrive/뉴스기사/정규화/'
# =======================================================================

# STEP 3: 출력 디렉토리 생성
# 저장할 폴더가 없으면 자동으로 생성합니다.
os.makedirs(output_directory, exist_ok=True)
print(f"결과가 저장될 폴더: {output_directory}")
print("-" * 50)

결과가 저장될 폴더: /content/drive/MyDrive/뉴스기사/정규화/
--------------------------------------------------


In [ ]:
try:
    # 입력 디렉토리 내의 모든 CSV 파일 목록 가져오기
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]

    if not csv_files:
        print(f"❌ 오류: '{input_directory}' 디렉토리에서 CSV 파일을 찾을 수 없습니다.")
    else:
        print(f"총 {len(csv_files)}개의 CSV 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            file_path = os.path.join(input_directory, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            # CSV 파일 읽기 (A, B, C, D 열로 가정)
            df = pd.read_csv(file_path)

            # 그룹화된 결과를 저장할 리스트
            result_rows = []
            i = 0
            while i < len(df):
                # D열이 0(개장일)이면 그룹 시작
                if df.iloc[i]['D'] == 0:
                    group_start_date = df.iloc[i]['A']
                    count_sum = df.iloc[i]['B']
                    days_list = [str(df.iloc[i]['C'])] # 요일을 문자열로 변환하여 리스트에 추가
                    num_days = 1
                    i += 1
                    # 다음 개장일이 나올 때까지 휴장일(D=1) 데이터 누적
                    while i < len(df) and df.iloc[i]['D'] == 1:
                        count_sum += df.iloc[i]['B']
                        days_list.append(str(df.iloc[i]['C']))
                        num_days += 1
                        i += 1

                    # 평균 기사 수 계산
                    avg_count = count_sum / num_days
                    # 요일 문자열 합치기
                    combined_days = ''.join(days_list)
                    result_rows.append([group_start_date, avg_count, combined_days])
                else:
                    i += 1

            # 결과 데이터프레임 생성
            result_df = pd.DataFrame(result_rows, columns=['A', 'B', 'F'])

            # B열(평균 기사 수)을 Min-Max 정규화하여 E열에 저장
            min_val = result_df['B'].min()
            max_val = result_df['B'].max()
            # 분모가 0이 되는 경우(모든 값이 같을 때)를 대비하여 예외 처리
            if (max_val - min_val) > 0:
                result_df['E'] = (result_df['B'] - min_val) / (max_val - min_val)
            else:
                result_df['E'] = 0 # 모든 값이 같으면 정규화 값은 0

            # 최종 열 순서 정리 (C, D열은 0으로 채움)
            result_df['C'] = 0
            result_df['D'] = 0
            final_df = result_df[['A', 'B', 'C', 'E', 'F', 'D']]

            # 새 파일명 생성 (예: '기아_sum_holiday_정규화.csv')
            name, ext = os.path.splitext(file_name)
            new_filename = f"{name}_정규화{ext}"
            output_path = os.path.join(output_directory, new_filename)

            # 새 CSV 파일로 저장
            final_df.to_csv(output_path, index=False)
            print(f"  - ✅ 완료! -> 저장된 파일: {new_filename}\n")

        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError:
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. '{input_directory}' 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


총 9개의 CSV 파일을 처리합니다.
--------------------------------------------------
🔄 처리 중인 파일: SK하이닉스_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: SK하이닉스_sum_holiday_정규화.csv

🔄 처리 중인 파일: POSCO홀딩스_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: POSCO홀딩스_sum_holiday_정규화.csv

🔄 처리 중인 파일: LG화학_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: LG화학_sum_holiday_정규화.csv

🔄 처리 중인 파일: KB금융_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: KB금융_sum_holiday_정규화.csv

🔄 처리 중인 파일: 네이버_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: 네이버_sum_holiday_정규화.csv

🔄 처리 중인 파일: 기아_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: 기아_sum_holiday_정규화.csv

🔄 처리 중인 파일: 삼성바이오로직스_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: 삼성바이오로직스_sum_holiday_정규화.csv

🔄 처리 중인 파일: 삼성전자_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: 삼성전자_sum_holiday_정규화.csv

🔄 처리 중인 파일: 현대차_sum_holiday.csv
  - ✅ 완료! -> 저장된 파일: 현대차_sum_holiday_정규화.csv

--------------------------------------------------
🎉 모든 파일 처리가 완료되었습니다!


파일이 1행이 abcde로 되면서 사라졌는데 1행이 1월1일이여서 괜찮을듯.

감성분석 과정


In [ ]:
# 원본 감성 분석 CSV 파일들이 있는 디렉토리 경로입니다.
source_dir = '/content/drive/MyDrive/result/first_file'
# 처리된 파일을 저장할 새 디렉토리 경로입니다.
dest_dir = '/content/drive/MyDrive/result/first_file/triple'

# 대상 디렉토리가 존재하지 않으면 생성합니다.
os.makedirs(dest_dir, exist_ok=True)
print(f"대상 디렉토리 '{dest_dir}'를 확인하거나 생성했습니다.")

대상 디렉토리 '/content/drive/MyDrive/result/first_file/triple'를 확인하거나 생성했습니다.


In [ ]:
# CSV 파일 목록 가져오기
# 원본 디렉토리에서 확장자가 '.csv'인 모든 파일을 찾습니다.
csv_files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]
print(f"'{source_dir}' 경로에서 {len(csv_files)}개의 CSV 파일을 찾았습니다.")

# 4. 각 CSV 파일 처리
# 찾은 각 CSV 파일을 반복하며 요청된 변환을 수행합니다.
for filename in csv_files:
    # 현재 처리 중인 파일의 전체 경로를 생성합니다.
    file_path = os.path.join(source_dir, filename)
    print(f"'{filename}' 파일을 처리 중입니다...")

    try:
        # CSV 파일을 DataFrame으로 읽어옵니다.
        # 첫 번째 행에 컬럼 이름이 있다고 가정합니다.
        df = pd.read_csv(file_path)

        # 필요한 컬럼 ('일자', '부정 확률 (0)', '긍정 확률 (1)')만 선택하여 복사합니다.
        # 이 컬럼 이름들은 업로드된 'Skhynix_sentiment_result.csv' 파일 스니펫을 기준으로 합니다.
        df_processed = df[['일자', '부정 확률 (0)', '긍정 확률 (1)']].copy()

        # '일자' 컬럼을 datetime 객체로 변환합니다.
        # 'YYYYMMDD' (예: 20250530) 형식이라고 가정하며, 오류 발생 시 NaT로 처리합니다.
        df_processed['일자'] = pd.to_datetime(df_processed['일자'], format='%Y%m%d', errors='coerce')

        # '일자' 컬럼 바로 뒤 (인덱스 1, 즉 B열 위치)에 '요일' 컬럼을 삽입합니다.
        # .dt.day_name()을 사용하여 영문 요일 이름을 추출합니다 (예: Monday, Tuesday).
        df_processed.insert(1, '요일', df_processed['일자'].dt.day_name())

        # '부정 확률 (0)'과 '긍정 확률 (1)' 컬럼의 값을 각각 3배로 만듭니다.
        df_processed['부정 확률 (0)'] = df_processed['부정 확률 (0)'] * 3
        df_processed['긍정 확률 (1)'] = df_processed['긍정 확률 (1)'] * 3

        # 원본 파일 이름에 '_triple.csv'를 붙여 새로운 파일 이름을 생성합니다.
        new_filename = filename.replace('.csv', '_triple.csv')
        # 처리된 파일을 저장할 새 파일의 전체 경로를 생성합니다.
        new_file_path = os.path.join(dest_dir, new_filename)

        # 처리된 DataFrame을 새 경로에 CSV 파일로 저장합니다.
        # index=False는 DataFrame 인덱스를 CSV에 쓰지 않도록 합니다.
        # encoding='utf-8-sig'는 한글 문자가 포함될 경우 깨지지 않도록 인코딩합니다.
        df_processed.to_csv(new_file_path, index=False, encoding='utf-8-sig')

        print(f"'{new_file_path}' 파일이 성공적으로 저장되었습니다.")

    except FileNotFoundError:
        print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    except KeyError as e:
        print(f"오류: '{filename}' 파일에 필요한 컬럼('일자', '부정 확률 (0)', '긍정 확률 (1)')이 없습니다. 누락된 컬럼: {e}")
    except Exception as e:
        print(f"'{filename}' 파일 처리 중 알 수 없는 오류 발생: {e}")

print("모든 파일 처리가 완료되었습니다. ✨")


'/content/drive/MyDrive/result/first_file' 경로에서 9개의 CSV 파일을 찾았습니다.
'Hyundai_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/Hyundai_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'Kbfg_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/Kbfg_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'Kia_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/Kia_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'Lgchem_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/Lgchem_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'Naver_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/Naver_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'PoscoHoldings_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/first_file/triple/PoscoHoldings_sentiment_result_triple.csv' 파일이 성공적으로 저장되었습니다.
'Samsung_sentiment_result.csv' 파일을 

In [ ]:
# 2. 경로 설정
# 원본 감성 분석 CSV 파일들이 있는 디렉토리 경로입니다.
# 변경됨: 'first_file'에서 'second_file'로 변경
source_dir = '/content/drive/MyDrive/result/second_file'
# 처리된 파일을 저장할 새 디렉토리 경로입니다.
# 변경됨: 'triple'에서 'second'로 변경
dest_dir = '/content/drive/MyDrive/result/second_file/second'

# 대상 디렉토리가 존재하지 않으면 생성합니다.
os.makedirs(dest_dir, exist_ok=True)
print(f"대상 디렉토리 '{dest_dir}'를 확인하거나 생성했습니다.")

대상 디렉토리 '/content/drive/MyDrive/result/second_file/second'를 확인하거나 생성했습니다.


In [ ]:
# 3. CSV 파일 목록 가져오기
# 원본 디렉토리에서 확장자가 '.csv'인 모든 파일을 찾습니다.
csv_files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]
print(f"'{source_dir}' 경로에서 {len(csv_files)}개의 CSV 파일을 찾았습니다.")

# 4. 각 CSV 파일 처리
# 찾은 각 CSV 파일을 반복하며 요청된 변환을 수행합니다.
for filename in csv_files:
    # 현재 처리 중인 파일의 전체 경로를 생성합니다.
    file_path = os.path.join(source_dir, filename)
    print(f"'{filename}' 파일을 처리 중입니다...")

    try:
        # CSV 파일을 DataFrame으로 읽어옵니다.
        # 첫 번째 행에 컬럼 이름이 있다고 가정합니다.
        df = pd.read_csv(file_path)

        # 필요한 컬럼 ('일자', '부정 확률 (0)', '긍정 확률 (1)')만 선택하여 복사합니다.
        # 이 컬럼 이름들은 업로드된 'Skhynix_sentiment_result.csv' 파일 스니펫을 기준으로 합니다.
        df_processed = df[['일자', '부정 확률 (0)', '긍정 확률 (1)']].copy()

        # '일자' 컬럼을 datetime 객체로 변환합니다.
        # 'YYYYMMDD' (예: 20250530) 형식이라고 가정하며, 오류 발생 시 NaT로 처리합니다.
        df_processed['일자'] = pd.to_datetime(df_processed['일자'], format='%Y%m%d', errors='coerce')

        # '일자' 컬럼 바로 뒤 (인덱스 1, 즉 B열 위치)에 '요일' 컬럼을 삽입합니다.
        # .dt.day_name()을 사용하여 영문 요일 이름을 추출합니다 (예: Monday, Tuesday).
        df_processed.insert(1, '요일', df_processed['일자'].dt.day_name())

        # '부정 확률 (0)'과 '긍정 확률 (1)' 컬럼의 값을 각각 1배로 만듭니다. (변경됨: 3배에서 1배로)
        df_processed['부정 확률 (0)'] = df_processed['부정 확률 (0)'] * 1
        df_processed['긍정 확률 (1)'] = df_processed['긍정 확률 (1)'] * 1

        # 원본 파일 이름에 '_ori.csv'를 붙여 새로운 파일 이름을 생성합니다. (변경됨: '_triple'에서 '_ori'로)
        new_filename = filename.replace('.csv', '_ori.csv')
        # 처리된 파일을 저장할 새 파일의 전체 경로를 생성합니다.
        new_file_path = os.path.join(dest_dir, new_filename)

        # 처리된 DataFrame을 새 경로에 CSV 파일로 저장합니다.
        # index=False는 DataFrame 인덱스를 CSV에 쓰지 않도록 합니다.
        # encoding='utf-8-sig'는 한글 문자가 포함될 경우 깨지지 않도록 인코딩합니다.
        df_processed.to_csv(new_file_path, index=False, encoding='utf-8-sig')

        print(f"'{new_file_path}' 파일이 성공적으로 저장되었습니다.")

    except FileNotFoundError:
        print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    except KeyError as e:
        print(f"오류: '{filename}' 파일에 필요한 컬럼('일자', '부정 확률 (0)', '긍정 확률 (1)')이 없습니다. 누락된 컬럼: {e}")
    except Exception as e:
        print(f"'{filename}' 파일 처리 중 알 수 없는 오류 발생: {e}")

print("모든 파일 처리가 완료되었습니다. ✨")


'/content/drive/MyDrive/result/second_file' 경로에서 9개의 CSV 파일을 찾았습니다.
'Hyundai_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/Hyundai_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'Kbfg_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/Kbfg_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'Kia_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/Kia_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'Lgchem_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/Lgchem_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'Naver_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/Naver_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'PoscoHoldings_sentiment_result.csv' 파일을 처리 중입니다...
'/content/drive/MyDrive/result/second_file/second/PoscoHoldings_sentiment_result_ori.csv' 파일이 성공적으로 저장되었습니다.
'Samsung_sentiment_result.csv' 파일을 처리 중입니다...


요일 합치고 평균


In [ ]:
# 2. 경로 설정
# 'second_file'에서 1배 처리된 파일들이 있는 디렉토리 경로입니다.
original_second_dir = '/content/drive/MyDrive/result/second_file/second'
# 'first_file'에서 3배 처리된 파일들이 있는 디렉토리 경로입니다.
original_triple_dir = '/content/drive/MyDrive/result/first_file/triple'

# 각 원본 디렉토리 내에 결과 파일을 저장할 'avg' 폴더 경로를 정의합니다.
second_avg_dir = os.path.join(original_second_dir, 'avg')
triple_avg_dir = os.path.join(original_triple_dir, 'avg')

# 'avg' 디렉토리가 존재하지 않으면 생성합니다.
os.makedirs(second_avg_dir, exist_ok=True)
os.makedirs(triple_avg_dir, exist_ok=True)
print(f"대상 디렉토리 '{second_avg_dir}'와 '{triple_avg_dir}'를 확인하거나 생성했습니다.")



대상 디렉토리 '/content/drive/MyDrive/result/second_file/second/avg'와 '/content/drive/MyDrive/result/first_file/triple/avg'를 확인하거나 생성했습니다.


In [ ]:

# 3. 각 디렉토리의 파일들을 개별적으로 처리하는 함수 정의
def process_and_average_files(source_directory, dest_directory): # suffix_to_remove 매개변수 제거
    """
    지정된 소스 디렉토리의 각 CSV 파일을 읽고, '일자' 기준으로 중복된 날짜의
    '부정 확률 (0)'과 '긍정 확률 (1)'을 평균내어 새로운 '_avg' 파일을 대상 디렉토리에 저장합니다.
    """
    # 모든 .csv 파일을 찾습니다.
    csv_files = [f for f in os.listdir(source_directory) if f.endswith('.csv')]
    print(f"'{source_directory}' 경로에서 {len(csv_files)}개의 CSV 파일을 찾았습니다.")


    for filename in csv_files:
        file_path = os.path.join(source_directory, filename)
        print(f"'{filename}' 파일 처리 중...")

        try:
            df = pd.read_csv(file_path)

            # '일자' 컬럼을 datetime 형식으로 변환합니다.
            # 변경된 부분: YYYY-MM-DD 형식으로 날짜를 파싱하도록 format을 변경합니다.
            df['일자'] = pd.to_datetime(df['일자'], format='%Y-%m-%d', errors='coerce')

            # '일자'와 '요일'을 기준으로 그룹화하고 '부정 확률 (0)', '긍정 확률 (1)'의 평균을 계산합니다.
            # 이는 같은 날짜에 여러 엔트리가 있을 경우를 처리하기 위함입니다.
            # '요일'은 '일자'에 종속되므로 함께 그룹화해도 무방합니다.
            averaged_df = df.groupby(['일자', '요일'], as_index=False)[
                ['부정 확률 (0)', '긍정 확률 (1)']
            ].mean()

            # '일자'를 다시 YYYYMMDD 형식의 정수로 변환하여 저장할 수 있도록 합니다.
            averaged_df['일자'] = averaged_df['일자'].dt.strftime('%Y%m%d').astype(int)

            # 새로운 파일 이름을 생성합니다 (원본 파일명에서 '.csv'만 제거하고 '_avg' 추가).
            output_filename = f"{filename.replace('.csv', '')}_avg.csv"
            output_path = os.path.join(dest_directory, output_filename)

            # 처리된 DataFrame을 새 경로에 CSV 파일로 저장합니다.
            averaged_df.to_csv(output_path, index=False, encoding='utf-8-sig')

            print(f"'{output_path}' 파일이 성공적으로 저장되었습니다. ✅")

        except KeyError as e:
            print(f"오류: '{filename}' 파일에 필요한 컬럼('일자', '요일', '부정 확률 (0)', '긍정 확률 (1)')이 없습니다. 누락된 컬럼: {e}")
        except Exception as e:
            print(f"'{filename}' 파일 처리 중 알 수 없는 오류 발생: {e}")
    print(f"'{source_directory}' 디렉토리의 모든 파일 처리가 완료되었습니다.")

# 4. 'second' 디렉토리의 파일들 처리
print("\n--- 'second' 디렉토리 파일 처리 시작 ---")
# 변경된 부분: process_and_average_files 함수 호출 시 suffix_to_remove 매개변수 제거
process_and_average_files(original_second_dir, second_avg_dir)

# 5. 'triple' 디렉토리의 파일들 처리
print("\n--- 'triple' 디렉토리 파일 처리 시작 ---")
# 변경된 부분: process_and_average_files 함수 호출 시 suffix_to_remove 매개변수 제거
process_and_average_files(original_triple_dir, triple_avg_dir)

print("\n모든 파일 처리가 완료되었습니다. ✨")


--- 'second' 디렉토리 파일 처리 시작 ---
'/content/drive/MyDrive/result/second_file/second' 경로에서 9개의 CSV 파일을 찾았습니다.
'Hyundai_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Hyundai_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Kbfg_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Kbfg_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Kia_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Kia_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Lgchem_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Lgchem_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Naver_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Naver_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'PoscoHoldings_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/PoscoHolding

In [ ]:
# 2. 경로 설정
# 'second_file'에서 1배 처리된 파일들이 있는 디렉토리 경로입니다.
original_second_dir = '/content/drive/MyDrive/share_drive/박박박사/emotion_analysis/second'
# 'first_file'에서 3배 처리된 파일들이 있는 디렉토리 경로입니다.
original_triple_dir = '/content/drive/MyDrive/share_drive/박박박사/emotion_analysis/triple'

# 각 원본 디렉토리 내에 결과 파일을 저장할 'avg' 폴더 경로를 정의합니다.
second_avg_dir = os.path.join(original_second_dir, 'avg')
triple_avg_dir = os.path.join(original_triple_dir, 'avg')

# 'avg' 디렉토리가 존재하지 않으면 생성합니다.
os.makedirs(second_avg_dir, exist_ok=True)
os.makedirs(triple_avg_dir, exist_ok=True)
print(f"대상 디렉토리 '{second_avg_dir}'와 '{triple_avg_dir}'를 확인하거나 생성했습니다.")



대상 디렉토리 '/content/drive/MyDrive/share_drive/박박박사/emotion_analysis/second/avg'와 '/content/drive/MyDrive/share_drive/박박박사/emotion_analysis/triple/avg'를 확인하거나 생성했습니다.


In [ ]:
# 3. 각 디렉토리의 파일들을 개별적으로 처리하는 함수 정의
def process_and_average_files(source_directory, dest_directory):
    """
    지정된 소스 디렉토리의 각 CSV 파일을 읽고, '일자' 기준으로 중복된 날짜의
    '부정 확률 (0)'과 '긍정 확률 (1)'을 평균내어 새로운 '_avg' 파일을 대상 디렉토리에 저장합니다.
    """
    # 모든 .csv 파일을 찾습니다.
    csv_files = [f for f in os.listdir(source_directory) if f.endswith('.csv')]
    print(f"'{source_directory}' 경로에서 {len(csv_files)}개의 CSV 파일을 찾았습니다.")


    for filename in csv_files:
        file_path = os.path.join(source_directory, filename)
        print(f"'{filename}' 파일 처리 중...")

        try:
            df = pd.read_csv(file_path)

            # '일자' 컬럼을 datetime 형식으로 변환합니다.
            # 변경된 부분: YYYY-MM-DD 형식으로 날짜를 파싱하도록 format을 변경합니다.
            df['일자'] = pd.to_datetime(df['일자'], format='%Y-%m-%d', errors='coerce')

            # '일자'와 '요일'을 기준으로 그룹화하고 '부정 확률 (0)', '긍정 확률 (1)'의 평균을 계산합니다.
            # 이는 같은 날짜에 여러 엔트리가 있을 경우를 처리하기 위함입니다.
            # '요일'은 '일자'에 종속되므로 함께 그룹화해도 무방합니다.
            averaged_df = df.groupby(['일자', '요일'], as_index=False)[
                ['부정 확률 (0)', '긍정 확률 (1)']
            ].mean()

            # '일자'를 다시 YYYYMMDD 형식의 정수로 변환하여 저장할 수 있도록 합니다.
            averaged_df['일자'] = averaged_df['일자'].dt.strftime('%Y%m%d').astype(int)

            # 새로운 파일 이름을 생성합니다 (원본 파일명에서 '.csv'만 제거하고 '_avg' 추가).
            output_filename = f"{filename.replace('.csv', '')}_avg.csv"
            output_path = os.path.join(dest_directory, output_filename)

            # 처리된 DataFrame을 새 경로에 CSV 파일로 저장합니다.
            averaged_df.to_csv(output_path, index=False, encoding='utf-8-sig')

            print(f"'{output_path}' 파일이 성공적으로 저장되었습니다. ✅")

        except KeyError as e:
            print(f"오류: '{filename}' 파일에 필요한 컬럼('일자', '요일', '부정 확률 (0)', '긍정 확률 (1)')이 없습니다. 누락된 컬럼: {e}")
        except Exception as e:
            print(f"'{filename}' 파일 처리 중 알 수 없는 오류 발생: {e}")
    print(f"'{source_directory}' 디렉토리의 모든 파일 처리가 완료되었습니다.")

# 4. 'second' 디렉토리의 파일들 처리
print("\n--- 'second' 디렉토리 파일 처리 시작 ---")
process_and_average_files(original_second_dir, second_avg_dir)

# 5. 'triple' 디렉토리의 파일들 처리
print("\n--- 'triple' 디렉토리 파일 처리 시작 ---")
process_and_average_files(original_triple_dir, triple_avg_dir)

# -----------------------------------------------------------------------
# 6. 'second/avg'와 'triple/avg' 폴더의 파일들을 같은 날짜끼리 합치고 단순 평균 계산
# -----------------------------------------------------------------------
print("\n--- 'second/avg'와 'triple/avg' 파일 합치기 및 단순 평균 계산 시작 ---")

# 최종 합산된 파일을 저장할 디렉토리 경로입니다.
sum_dir = os.path.join('/content/drive/MyDrive/result', 'sum')

# 'sum' 디렉토리가 존재하지 않으면 생성합니다.
os.makedirs(sum_dir, exist_ok=True)
print(f"대상 디렉토리 '{sum_dir}'를 확인하거나 생성했습니다.")

# 각 디렉토리의 CSV 파일 목록 가져오기 및 매핑
# 파일명에서 회사명 (예: Skhynix_sentiment_result)을 추출하여 맵을 생성합니다.
def get_file_map_for_sum(directory, suffix_to_remove):
    file_map = {}
    csv_files = [f for f in os.listdir(directory) if f.endswith(suffix_to_remove)]
    for f in csv_files:
        # 파일명에서 접미사를 제거하여 기본 회사 이름을 추출합니다.
        # 예: Skhynix_sentiment_result_ori_avg.csv -> Skhynix_sentiment_result
        base_company_name = f.replace(suffix_to_remove, '')
        file_map[base_company_name] = os.path.join(directory, f)
    return file_map

# '_ori_avg.csv' 파일을 포함하는 맵 생성 (second/avg 폴더)
second_avg_files_map = get_file_map_for_sum(second_avg_dir, '_ori_avg.csv')
print(f"'{second_avg_dir}'에서 {len(second_avg_files_map)}개의 '_ori_avg.csv' 파일을 찾았습니다.")

# '_triple_avg.csv' 파일을 포함하는 맵 생성 (triple/avg 폴더)
triple_avg_files_map = get_file_map_for_sum(triple_avg_dir, '_triple_avg.csv')
print(f"'{triple_avg_dir}'에서 {len(triple_avg_files_map)}개의 '_triple_avg.csv' 파일을 찾았습니다.")

# 파일 쌍 합치고 단순 평균 계산
processed_sum_count = 0
# second_avg_files_map에 있는 회사 이름을 기준으로 반복합니다.
for base_company_name in second_avg_files_map:
    # 해당하는 파일이 triple_avg_files_map에도 있는지 확인합니다.
    if base_company_name in triple_avg_files_map:
        processed_sum_count += 1
        file_path_second = second_avg_files_map[base_company_name]
        file_path_triple = triple_avg_files_map[base_company_name]
        print(f"🔄 '{base_company_name}' 회사 파일 쌍 합치기 및 평균 계산 시작...")

        try:
            # 두 DataFrame을 읽어옵니다.
            df_second = pd.read_csv(file_path_second)
            df_triple = pd.read_csv(file_path_triple)

            # '일자' 컬럼을 datetime 형식으로 변환합니다.
            df_second['일자'] = pd.to_datetime(df_second['일자'].astype(str), format='%Y%m%d', errors='coerce')
            df_triple['일자'] = pd.to_datetime(df_triple['일자'].astype(str), format='%Y%m%d', errors='coerce')

            # '일자'를 기준으로 외부 조인합니다.
            merged_df = pd.merge(
                df_second, df_triple, on='일자', how='outer', suffixes=('_second', '_triple')
            )

            # '요일' 컬럼은 어느 한쪽에서 가져오거나, NaN이 아닌 값으로 채웁니다.
            merged_df['요일'] = merged_df['요일_second'].fillna(merged_df['요일_triple'])

            # 부정 확률 (0) 및 긍정 확률 (1) 컬럼을 단순 합산 후 4로 나눕니다.
            # (각 파일의 값 + 다른 파일의 값) / 4
            # 변경된 부분: 가중치 *1과 *3 제거, 단순히 두 값을 더하고 4로 나눔
            merged_df['부정 확률 (0)'] = (
                merged_df['부정 확률 (0)_second'].fillna(0) +
                merged_df['부정 확률 (0)_triple'].fillna(0)
            ) / 4

            merged_df['긍정 확률 (1)'] = (
                merged_df['긍정 확률 (1)_second'].fillna(0) +
                merged_df['긍정 확률 (1)_triple'].fillna(0)
            ) / 4

            # 최종 DataFrame에 필요한 컬럼만 선택하고, '일자' 기준으로 정렬합니다.
            final_df = merged_df[['일자', '요일', '부정 확률 (0)', '긍정 확률 (1)']].sort_values(by='일자').reset_index(drop=True)

            # '일자'를 다시 YYYYMMDD 형식의 정수로 변환하여 저장할 수 있도록 합니다.
            final_df['일자'] = final_df['일자'].dt.strftime('%Y%m%d').astype(int)

            # 저장할 파일명 생성 (예: 'Skhynix_sentiment_result_sum.csv')
            output_filename = f"{base_company_name}_sum.csv"
            output_path = os.path.join(sum_dir, output_filename)

            # CSV 파일로 저장
            final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"  - ✅ 완료! -> 저장된 파일: {output_filename}\n")

        except KeyError as e:
            print(f"  - ❌ 오류: '{base_company_name}'의 파일 중 하나에 필요한 컬럼이 없습니다. 누락된 컬럼: {e}\n")
        except Exception as e:
            print(f"  - ❌ 알 수 없는 오류가 발생했습니다: {e}\n")
    else:
        print(f"  - ⚠️ 경고: 짝이 되는 '_triple_avg.csv' 파일을 찾을 수 없어 '{base_company_name}' 건너뜁니다.\n")

print("-" * 50)
if processed_sum_count > 0:
    print(f"🎉 총 {processed_sum_count}개의 파일 쌍 처리가 완료되었습니다!")
else:
    print("❌ 처리된 파일 쌍이 없습니다. 파일 이름과 경로를 다시 확인해주세요.")



--- 'second' 디렉토리 파일 처리 시작 ---
'/content/drive/MyDrive/result/second_file/second' 경로에서 9개의 CSV 파일을 찾았습니다.
'Hyundai_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Hyundai_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Kbfg_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Kbfg_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Kia_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Kia_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Lgchem_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Lgchem_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'Naver_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/Naver_sentiment_result_ori_avg.csv' 파일이 성공적으로 저장되었습니다. ✅
'PoscoHoldings_sentiment_result_ori.csv' 파일 처리 중...
'/content/drive/MyDrive/result/second_file/second/avg/PoscoHolding

In [ ]:
# =======================================================================
# ✅ 사용자 설정 영역
# 입력 폴더와 출력 폴더 경로를 설정합니다.
# =======================================================================
# 합산된 감성 데이터(_sum.csv 파일들)가 있는 폴더
input_directory = '/content/drive/MyDrive/result/sum'
# 최종 휴장일 처리 및 정규화된 파일을 저장할 폴더
output_directory = os.path.join('/content/drive/MyDrive/result', 'holiday_norm')

# 추가됨: 요일 및 휴장일 정보가 추가된 중간 파일을 저장할 폴더
holiday_output_directory = os.path.join('/content/drive/MyDrive/result', 'holiday')
# =======================================================================


In [ ]:
# 2. 휴장일 정보 설정 (2025년 기준)
# 주말(토, 일)은 코드에서 자동으로 휴장일 처리됩니다.
holiday_set = set([
   '2018-01-01',
    '2018-02-15', '2018-02-16',
    '2018-03-01',
    '2018-05-01',
    '2018-05-07',
    '2018-05-22',
    '2018-06-06', '2018-06-13',
    '2018-08-15',
    '2018-09-24', '2018-09-25', '2018-09-26',
    '2018-10-03', '2018-10-09',
    '2018-12-25', '2018-12-31',
    '2019-01-01',
    '2019-02-04', '2019-02-05', '2019-02-06'
])

# 결과가 저장될 폴더 생성
os.makedirs(output_directory, exist_ok=True)
# 추가됨: 중간 휴장일 파일 저장 폴더 생성
os.makedirs(holiday_output_directory, exist_ok=True)
print(f"최종 결과가 저장될 폴더: {output_directory}")
print(f"중간 휴장일 파일이 저장될 폴더: {holiday_output_directory}")
print("-" * 50)



최종 결과가 저장될 폴더: /content/drive/MyDrive/result/holiday_norm
중간 휴장일 파일이 저장될 폴더: /content/drive/MyDrive/result/holiday
--------------------------------------------------


In [ ]:
try: # 이 try 블록은 전체 파일 처리 과정을 포괄합니다.
    # 입력 디렉토리 내의 모든 CSV 파일 목록 가져오기
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('_sum.csv')] # _sum.csv 파일만 대상으로 함

    if not csv_files:
        print(f"❌ 오류: '{input_directory}' 디렉토리에서 '_sum.csv' 파일을 찾을 수 없습니다.")
    else: # 파일을 찾았을 경우의 로직 시작
        print(f"총 {len(csv_files)}개의 '_sum.csv' 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            file_path = os.path.join(input_directory, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            try: # 이 try 블록은 개별 파일 처리 중 발생할 수 있는 오류를 잡습니다.
                # CSV 파일 읽기
                # '일자', '요일', '부정 확률 (0)', '긍정 확률 (1)' 컬럼을 가집니다.
                df = pd.read_csv(file_path)

                # '일자'열을 datetime 형식으로 변환 (YYYYMMDD 포맷을 명시적으로 지정)
                df['일자'] = pd.to_datetime(df['일자'], format='%Y%m%d', errors='coerce')

                # '휴장일여부' 컬럼 추가
                df['휴장일여부'] = df['일자'].apply(
                    lambda x: 1 if pd.isnull(x) or (x.strftime('%Y-%m-%d') in holiday_set or x.weekday() >= 5) else 0
                )

                # --- 휴장일 정보가 추가된 파일 저장 ---
                # '일자' 컬럼을 다시 YYYYMMDD 형식의 정수로 변환하여 저장
                df_for_holiday_output = df.copy()
                df_for_holiday_output['일자'] = df_for_holiday_output['일자'].dt.strftime('%Y%m%d').astype(int)

                # 새 파일명 생성 (예: 'KB금융_sum_holiday.csv')
                name, ext = os.path.splitext(file_name)
                new_holiday_filename = f"{name}_holiday{ext}"
                holiday_output_path = os.path.join(holiday_output_directory, new_holiday_filename)

                # 새 CSV 파일로 저장
                # 원본 _sum.csv의 컬럼들과 새로 추가된 휴장일여부 컬럼을 포함
                df_for_holiday_output.to_csv(holiday_output_path, index=False, encoding='utf-8-sig')
                print(f"  - ✅ 완료! -> 저장된 파일: {new_holiday_filename}\n")
                # --- 휴장일 정보가 추가된 파일 저장 끝 ---

            except KeyError as e: # 개별 파일 처리 중 컬럼이 없을 때의 오류
                print(f"❌ 오류: '{file_name}' 파일에 필요한 컬럼이 없습니다. 누락된 컬럼: {e}")
            except Exception as e: # 개별 파일 처리 중 기타 오류
                print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")

        # 모든 파일 처리 완료 메시지 (for 루프 끝난 후)
        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError: # 입력 디렉토리를 찾을 수 없을 때의 오류
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. '{input_directory}' 경로를 다시 확인해주세요.")
except Exception as e: # 전체 프로세스 중 발생할 수 있는 기타 오류
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


총 9개의 '_sum.csv' 파일을 처리합니다.
--------------------------------------------------
🔄 처리 중인 파일: Hyundai_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Hyundai_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Kbfg_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Kbfg_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Kia_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Kia_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Lgchem_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Lgchem_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Naver_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Naver_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: PoscoHoldings_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: PoscoHoldings_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Samsung_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Samsung_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: SamsungBiologics_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: SamsungBiologics_sentiment_result_sum_holiday.csv

🔄 처리 중인 파일: Skhynix_sentiment_result_sum.csv
  

In [ ]:
# =======================================================================
# ✅ 사용자 설정 영역
# 입력 폴더와 출력 폴더 경로를 설정합니다.
# =======================================================================
# 합산된 감성 데이터(_sum.csv 파일들)가 있는 폴더
input_directory = '/content/drive/MyDrive/result/sum'
# 최종 휴장일 처리된 파일을 저장할 폴더 (정규화는 제외됨)
output_directory = os.path.join('/content/drive/MyDrive/result/', 'holiday_processed')
# 'holiday_output_directory' 관련 코드 제거됨
# =======================================================================

# 2. 휴장일 정보 설정 (2025년 기준)
# 주말(토, 일)은 코드에서 자동으로 휴장일 처리됩니다.
holiday_set = set([
    '2018-01-01',
    '2018-02-15', '2018-02-16',
    '2018-03-01',
    '2018-05-01',
    '2018-05-07',
    '2018-05-22',
    '2018-06-06', '2018-06-13',
    '2018-08-15',
    '2018-09-24', '2018-09-25', '2018-09-26',
    '2018-10-03', '2018-10-09',
    '2018-12-25', '2018-12-31',
    '2019-01-01',
    '2019-02-04', '2019-02-05', '2019-02-06'
])

# 결과가 저장될 폴더 생성
os.makedirs(output_directory, exist_ok=True)
# 'holiday_output_directory' 폴더 생성 관련 코드 제거됨
print(f"최종 결과가 저장될 폴더: {output_directory}")
# 'holiday_output_directory' 관련 print 제거됨
print("-" * 50)

try: # 이 try 블록은 전체 파일 처리 과정을 포괄합니다.
    # 입력 디렉토리 내의 모든 CSV 파일 목록 가져오기
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('_sum.csv')] # _sum.csv 파일만 대상으로 함

    if not csv_files:
        print(f"❌ 오류: '{input_directory}' 디렉토리에서 '_sum.csv' 파일을 찾을 수 없습니다.")
    else: # 파일을 찾았을 경우의 로직 시작
        print(f"총 {len(csv_files)}개의 '_sum.csv' 파일을 처리합니다.")
        print("-" * 50)

        # 각 CSV 파일에 대해 반복 작업 수행
        for file_name in csv_files:
            file_path = os.path.join(input_directory, file_name)
            print(f"🔄 처리 중인 파일: {file_name}")

            try: # 이 try 블록은 개별 파일 처리 중 발생할 수 있는 오류를 잡습니다.
                # CSV 파일 읽기
                # '일자', '요일', '부정 확률 (0)', '긍정 확률 (1)' 컬럼을 가집니다.
                df = pd.read_csv(file_path)

                # '일자'열을 datetime 형식으로 변환 (YYYYMMDD 포맷을 명시적으로 지정)
                df['일자'] = pd.to_datetime(df['일자'], format='%Y%m%d', errors='coerce')

                # '휴장일여부' 컬럼 추가
                df['휴장일여부'] = df['일자'].apply(
                    lambda x: 1 if pd.isnull(x) or (x.strftime('%Y-%m-%d') in holiday_set or x.weekday() >= 5) else 0
                )

                # --- 이전의 중간 파일 저장 부분 제거됨 ---
                # 'df_for_holiday_output' 생성 및 'to_csv' 호출 부분 제거됨


                # --- 휴장일 데이터 합치기 및 평균 계산 로직 (직전 개장일에 소급 적용) ---
                # temp_df는 휴장일여부까지 추가된 DataFrame
                temp_df = df.copy()

                # 데이터를 역순으로 순회하며 휴장일 정보를 직전 개장일에 롤백
                current_rollback_neg_sum = 0
                current_rollback_pos_sum = 0
                current_rollback_day_count = 0
                current_rollback_days_list = []

                for i in range(len(temp_df) - 1, -1, -1):
                    row_index = i
                    current_row = temp_df.iloc[row_index]

                    if current_row['휴장일여부'] == 1: # 휴장일인 경우
                        current_rollback_neg_sum += current_row['부정 확률 (0)']
                        current_rollback_pos_sum += current_row['긍정 확률 (1)']
                        current_rollback_day_count += 1
                        current_rollback_days_list.insert(0, str(current_row['요일']))
                    else: # 개장일인 경우 (또는 현재가 첫 행)
                        total_neg_sum = current_row['부정 확률 (0)'] + current_rollback_neg_sum
                        total_pos_sum = current_row['긍정 확률 (1)'] + current_rollback_pos_sum
                        total_day_count = 1 + current_rollback_day_count

                        avg_neg_prob = total_neg_sum / total_day_count
                        avg_pos_prob = total_pos_sum / total_day_count

                        temp_df.at[row_index, '부정 확률 (0)'] = avg_neg_prob
                        temp_df.at[row_index, '긍정 확률 (1)'] = avg_pos_prob

                        combined_day_str = "".join([str(current_row['요일'])] + current_rollback_days_list)
                        temp_df.at[row_index, '그룹_요일_요약'] = combined_day_str

                        current_rollback_neg_sum = 0
                        current_rollback_pos_sum = 0
                        current_rollback_day_count = 0
                        current_rollback_days_list = []

                # 최종 결과는 휴장일이 아닌 행만 선택 (정규화 제외)
                processed_df = temp_df[temp_df['휴장일여부'] == 0].copy()

                # --- Min-Max 정규화 로직 제거됨 ---
                # '감성 점수' 및 '정규화된 감성 점수 (E)' 컬럼 생성 및 계산 로직을 제거했습니다.

                # 최종 열 순서 정리 (정규화된 감성 점수 (E) 컬럼 제외)
                # A(일자), B(요일), C(부정 확률 (0)), D(긍정 확률 (1)), E(휴장일여부), F(그룹_요일_요약)
                final_df = processed_df[['일자', '요일', '부정 확률 (0)', '긍정 확률 (1)', '휴장일여부', '그룹_요일_요약']].copy()

                # '일자'를 다시 YYYYMMDD 형식의 정수로 변환하여 저장
                final_df['일자'] = final_df['일자'].dt.strftime('%Y%m%d').astype(int)

                # 새 파일명 생성 (예: 'KB금융_sum_holiday_processed.csv')
                # '정규화'가 빠졌으므로 파일명도 'holiday_processed'로 변경 제안
                name, ext = os.path.splitext(file_name)
                new_filename = f"{name}_holiday_processed{ext}"
                output_path = os.path.join(output_directory, new_filename)

                # 새 CSV 파일로 저장
                final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
                print(f"  - ✅ 완료! -> 저장된 파일: {new_filename}\n")

            except KeyError as e: # 개별 파일 처리 중 컬럼이 없을 때의 오류
                print(f"❌ 오류: '{file_name}' 파일에 필요한 컬럼이 없습니다. 누락된 컬럼: {e}")
            except Exception as e: # 개별 파일 처리 중 기타 오류
                print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")

        # 모든 파일 처리 완료 메시지 (for 루프 끝난 후)
        print("-" * 50)
        print("🎉 모든 파일 처리가 완료되었습니다!")

except FileNotFoundError: # 입력 디렉토리를 찾을 수 없을 때의 오류
    print(f"❌ 오류: 입력 디렉토리를 찾을 수 없습니다. '{input_directory}' 경로를 다시 확인해주세요.")
except Exception as e: # 전체 프로세스 중 발생할 수 있는 기타 오류
    print(f"❌ 알 수 없는 오류가 발생했습니다: {e}")


최종 결과가 저장될 폴더: /content/drive/MyDrive/result/holiday_processed
--------------------------------------------------
총 9개의 '_sum.csv' 파일을 처리합니다.
--------------------------------------------------
🔄 처리 중인 파일: Hyundai_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Hyundai_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: Kbfg_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Kbfg_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: Kia_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Kia_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: Lgchem_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Lgchem_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: Naver_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Naver_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: PoscoHoldings_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: PoscoHoldings_sentiment_result_sum_holiday_processed.csv

🔄 처리 중인 파일: Samsung_sentiment_result_sum.csv
  - ✅ 완료! -> 저장된 파일: Samsung_sentiment_result_sum_holiday_proc